In [1]:
import sys
print(sys.executable)

/data/home/tkodippili/Desktop/localTest_Analysis_DashboardV3/Dashboard/server/.venv/bin/python


In [3]:
from pathlib import Path
import importlib
import csv

import pandas as pd


def _find_repo_root() -> Path:
    """Resolve repo root so data/ paths work regardless of notebook CWD."""
    start = Path.cwd().resolve()
    for d in [start, *start.parents]:
        if (d / "server" / "pyproject.toml").is_file():
            return d
    return start


def _resolve_rpc_reader_ctor():
    """Return a callable constructor from rpc_reader package variants."""
    rr = importlib.import_module("rpc_reader")

    direct_names = ("ReadRPC", "RPCReader")
    for name in direct_names:
        ctor = getattr(rr, name, None)
        if callable(ctor):
            return ctor

    for submodule in ("rpc_reader", "reader", "core"):
        try:
            mod = importlib.import_module(f"rpc_reader.{submodule}")
        except Exception:
            continue
        for name in direct_names:
            ctor = getattr(mod, name, None)
            if callable(ctor):
                return ctor

    visible = [name for name in dir(rr) if not name.startswith("_")]
    raise AttributeError(
        "Installed 'rpc_reader' module has no supported reader class. "
        f"module={getattr(rr, '__file__', '<unknown>')} attrs={visible[:25]}"
    )


def _to_header_dict(obj):
    """Normalize header metadata into a dict without dropping keys."""
    if obj is None:
        return {}
    if isinstance(obj, dict):
        return {str(k): v for k, v in obj.items()}
    if isinstance(obj, list):
        out = {}
        for i, item in enumerate(obj):
            out[f"item_{i}"] = item
        return out
    return {"raw_header": obj}


def _extract_channel_names(channels, width):
    name_keys = (
        "Description",
        "description",
        "DESC",
        "title",
        "Title",
        "name",
        "Name",
        "channel_name",
    )
    names = []
    for i in range(width):
        ch = channels[i] if i < len(channels) else None

        name = None
        if isinstance(ch, str):
            name = ch
        elif isinstance(ch, dict):
            for key in name_keys:
                if ch.get(key):
                    name = ch[key]
                    break
        elif ch is not None:
            for key in name_keys:
                value = getattr(ch, key, None)
                if value:
                    name = value
                    break

        names.append(str(name).strip() if name else f"channel_{i + 1}")

    # Ensure headers are unique and preserve full width.
    seen = {}
    unique = []
    for n in names:
        count = seen.get(n, 0)
        seen[n] = count + 1
        unique.append(n if count == 0 else f"{n}_{count + 1}")
    return unique


def _load_via_rpc_reader(rsp_path: Path):
    ctor = _resolve_rpc_reader_ctor()
    reader = ctor(rsp_path)

    for load_method in ("import_rpc_data_from_file", "read", "load", "parse"):
        fn = getattr(reader, load_method, None)
        if callable(fn):
            fn()
            break

    data = None
    get_data = getattr(reader, "get_data", None)
    if callable(get_data):
        data = get_data()
    elif hasattr(reader, "data"):
        data = reader.data

    if data is None:
        raise RuntimeError("rpc_reader backend could not extract numeric channel data")

    channels = []
    get_channels = getattr(reader, "get_channels", None)
    if callable(get_channels):
        channels = get_channels() or []
    elif hasattr(reader, "channels"):
        channels = reader.channels or []

    headers = None
    get_headers = getattr(reader, "get_headers", None)
    if callable(get_headers):
        headers = get_headers()
    elif hasattr(reader, "headers"):
        headers = reader.headers

    return data, channels, _to_header_dict(headers)


def _load_via_pyrpc3(rsp_path: Path):
    from pyRPC3 import RPC3

    rpc = RPC3(str(rsp_path), debug=False)

    channels = getattr(rpc, "channels", []) or []
    if channels:
        series = []
        for ch in channels:
            values = (
                getattr(ch, "data", None)
                or getattr(ch, "values", None)
                or getattr(ch, "y", None)
            )
            values = list(values) if values is not None else []
            series.append(values)

        max_len = max((len(col) for col in series), default=0)
        padded = [col + [None] * (max_len - len(col)) for col in series]
        df = pd.DataFrame(padded).T
        df.columns = _extract_channel_names(channels, df.shape[1])

        headers = {
            "backend": "pyRPC3",
            "channel_count": len(channels),
            "row_count": len(df),
        }
        return df, headers

    matrix = getattr(rpc, "data", None)
    if matrix is None:
        raise RuntimeError("pyRPC3 backend did not expose channel data")

    df = pd.DataFrame(matrix)
    df.columns = _extract_channel_names([], df.shape[1])
    headers = {
        "backend": "pyRPC3",
        "channel_count": df.shape[1],
        "row_count": len(df),
    }
    return df, headers


def _extract_channel_units(channels, width):
    units = []
    for i in range(width):
        ch = channels[i] if i < len(channels) else None
        unit = None

        if isinstance(ch, dict):
            unit = ch.get("units") or ch.get("Units")
        elif ch is not None and not isinstance(ch, str):
            unit = getattr(ch, "units", None) or getattr(ch, "unit", None)

        units.append(str(unit).strip() if unit else "")
    return units


def _resolve_dt(header_dict):
    for key in (
        "DELTA_T",
        "DELTA T",
        "DELTAT",
        "delta_t",
        "sample_interval",
        "x_increment",
    ):
        if key in header_dict:
            try:
                return float(header_dict[key])
            except Exception:
                pass
    return 1.0


def _load_reference_titles_if_present(rsp_path: Path, expected_count: int):
    """Reuse channel titles from a known reference export when available."""
    candidates = [
        rsp_path.with_name(f"{rsp_path.stem}_GND.csv"),
        rsp_path.with_name("gnd_comp.csv"),
        rsp_path.with_name(f"{rsp_path.stem}_simultaneous_values00 copy.csv"),
        rsp_path.with_name(f"{rsp_path.stem}_simultaneous_values00.csv"),
    ]

    for cand in candidates:
        if not cand.exists():
            continue
        try:
            with cand.open("r", encoding="utf-8", newline="") as f:
                rows = list(csv.reader(f))

            for idx, row in enumerate(rows[:-1]):
                if not row or row[0] != "#TITLES":
                    continue

                title_row = rows[idx + 1]
                if len(title_row) >= expected_count + 2:
                    raw_titles = title_row[2 : 2 + expected_count]
                    if len(raw_titles) == expected_count:
                        return raw_titles

                if len(title_row) >= expected_count + 7:
                    feature_titles = title_row[7 : 7 + expected_count]
                    if len(feature_titles) == expected_count:
                        return feature_titles
        except Exception:
            continue

    return None


def rsp_to_tagged_csv(rsp_path, csv_path=None):
    rsp_path = Path(rsp_path)
    csv_path = Path(csv_path) if csv_path else rsp_path.with_suffix(".csv")

    backend_errors = []
    df = None
    header_dict = {}
    channels = []

    try:
        data, channels, headers = _load_via_rpc_reader(rsp_path)
        df = pd.DataFrame(data)
        df.columns = _extract_channel_names(channels, df.shape[1])
        header_dict = headers
        header_dict["backend"] = "rpc_reader"
        print("Backend: rpc_reader")
    except SystemExit as exc:
        # rpc_reader uses sys.exit() for validation errors (not a subclass of Exception).
        msg = exc.args[0] if exc.args else str(exc)
        backend_errors.append(f"rpc_reader failed: {msg}")
    except Exception as exc:
        backend_errors.append(f"rpc_reader failed: {exc}")

    if df is None:
        try:
            df, header_dict = _load_via_pyrpc3(rsp_path)
            channels = []
            header_dict["backend"] = "pyRPC3"
            print("Backend: pyRPC3")
        except SystemExit as exc:
            msg = exc.args[0] if exc.args else str(exc)
            backend_errors.append(f"pyRPC3 failed: {msg}")
        except Exception as exc:
            backend_errors.append(f"pyRPC3 failed: {exc}")

    if df is None:
        msg = "\n".join(backend_errors)
        raise RuntimeError(
            "No supported RSP parser worked. Install one of:\n"
            "  - python3 -m pip install rpc-reader\n"
            "  - python3 -m pip install git+https://github.com/galuszkm/pyRPC3.git\n"
            f"Details:\n{msg}"
        )

    channel_names = list(df.columns)
    titled_names = [f"{i + 1} {name}" for i, name in enumerate(channel_names)]

    # Last-resort fallback only when the RSP metadata lacks channel descriptions.
    if all(name.startswith("channel_") for name in channel_names):
        ref_titles = _load_reference_titles_if_present(rsp_path, len(channel_names))
        if ref_titles:
            titled_names = ref_titles

    units = _extract_channel_units(channels, len(channel_names))
    dt = _resolve_dt(header_dict)

    with csv_path.open("w", newline="", encoding="utf-8") as f:
        writer = csv.writer(f)

        writer.writerow(["#HEADER"])
        writer.writerow(["#TITLES"])
        writer.writerow(["", ""] + titled_names)
        writer.writerow(["#UNITS"])
        writer.writerow(["", ""] + units)
        writer.writerow(["#DATATYPES"])
        writer.writerow(["Huge", "Double"] + ["Float"] * len(channel_names))
        writer.writerow(["#DATA"])

        for row_idx, values in enumerate(df.itertuples(index=False, name=None), start=1):
            time_value = f"{(row_idx - 1) * dt:.6f}"
            writer.writerow([row_idx, time_value] + list(values))

    return csv_path, df.shape, len(channel_names), len(header_dict)


rsp_path = _find_repo_root() / "tests/upload_data/4e1_bt1xx_bt1tc_vpi_701_20240828_lca_lf.rsp"
if not rsp_path.is_file():
    raise FileNotFoundError(
        f"RSP not found: {rsp_path}\n"
        f"cwd={Path.cwd()!s} repo_root={_find_repo_root()!s}"
    )
out, shape, channel_count, header_count = rsp_to_tagged_csv(rsp_path)
print(
    f"Converted: {rsp_path} -> {out} shape={shape} "
    f"channels={channel_count} headers_saved={header_count}"
)

 Reading headers from:
	File: /data/home/tkodippili/Desktop/localTest_Analysis_DashboardV3/Dashboard/tests/upload_data/4e1_bt1xx_bt1tc_vpi_701_20240828_lca_lf.rsp
 Data structure summery:
	Channels to read:  21
	Points per frame:  1024
	Points per group:  2048
	Number of frames:  6
	Number of groups:  3
	Header end at:     18432 bytes
	File end at:       276480
	Bytes to read:     258048
 Frame grouping array:
 [[1, 2], [3, 4], [5, 6]]
 Binary decoding settings: <1024h, 2048,  Bytes per data value: 2

 Reading test data from 21 channels,
	Progress: |██████████████████████████████████████████████████| 100.0% Complete
Backend: rpc_reader
Converted: /data/home/tkodippili/Desktop/localTest_Analysis_DashboardV3/Dashboard/tests/upload_data/4e1_bt1xx_bt1tc_vpi_701_20240828_lca_lf.rsp -> /data/home/tkodippili/Desktop/localTest_Analysis_DashboardV3/Dashboard/tests/upload_data/4e1_bt1xx_bt1tc_vpi_701_20240828_lca_lf.csv shape=(6144, 21) channels=21 headers_saved=145
